# Milestones 11–12 — Locked final evaluation and efficiency

This notebook accepts only the final bundle containing the already-frozen development selection. It caches Qwen test logits resumably, evaluates all six models once on the public test set, and benchmarks Qwen against the selected distilled student on the same GPU.

In [ ]:
from google.colab import drive, files
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile
from io import BytesIO
import hashlib
import json
import subprocess
import sys

drive.mount('/content/drive')
uploaded = files.upload()  # Select finevid_final_source.zip.
if len(uploaded) != 1:
    raise ValueError('Upload exactly one final source bundle.')
bundle_name, bundle_bytes = next(iter(uploaded.items()))
bundle_hash = hashlib.sha256(bundle_bytes).hexdigest()
PROJECT_DIR = Path('/content') / ('finevid-distill-final-' + bundle_hash[:12])
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
with ZipFile(BytesIO(bundle_bytes)) as archive:
    manifest = json.loads(archive.read('bundle_manifest.json'))
    if manifest.get('test_access_gate') != 'frozen final selection':
        raise ValueError('This is not the locked final-evaluation bundle.')
    expected = set(manifest['files']) | {'bundle_manifest.json'}
    if len(archive.namelist()) != len(expected) or set(archive.namelist()) != expected:
        raise ValueError('Bundle file list does not match its manifest.')
    for name, expected_hash in manifest['files'].items():
        destination = (PROJECT_DIR / name).resolve()
        if not destination.is_relative_to(PROJECT_DIR.resolve()):
            raise ValueError('Invalid archive path.')
        if hashlib.sha256(archive.read(name)).hexdigest() != expected_hash:
            raise ValueError('Bundle integrity check failed: ' + name)
    archive.extractall(PROJECT_DIR)
del uploaded, bundle_bytes
ARTIFACT_ROOT = Path('/content/drive/MyDrive/FinEvid-Distill')
TEACHER_DIR = ARTIFACT_ROOT / 'teacher_scores'
CHECKPOINT_ROOT = ARTIFACT_ROOT / 'checkpoints'
HARD_DIR = CHECKPOINT_ROOT / 'hard_label_student_tau005'
DISTILLED_DIR = CHECKPOINT_ROOT / 'distilled_student'
RESULT_DIR = ARTIFACT_ROOT / 'experiment_results'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
SELECTION = PROJECT_DIR / 'outputs/final_selection.json'
TEST_DATA = PROJECT_DIR / 'data/processed/test.jsonl'
print('Verified locked final bundle:', bundle_hash)
print('Frozen selection:', SELECTION)

## Install, verify the GPU, and run all tests

In [ ]:
%pip install -q -r {PROJECT_DIR / 'requirements-colab.txt'}
%pip install -q -e {PROJECT_DIR}

In [ ]:
def run_project(*arguments):
    return subprocess.run([sys.executable, '-u', *arguments], cwd=PROJECT_DIR, check=True)

run_project('-c', "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.cuda.get_device_name(0)); print('PyTorch:', torch.__version__)")
run_project('-m', 'pytest', '-q')

## Cache the Qwen public-test scores

The command validates the frozen selection against both completed development-selected runs before touching the test split. Its `.partial` file resumes by whole question after a disconnect.

In [ ]:
run_project(
    'src/data/cache_teacher_scores.py',
    '--splits', 'test',
    '--processed-dir', str(TEST_DATA.parent),
    '--output-dir', str(TEACHER_DIR),
    '--final-selection', str(SELECTION),
    '--hard-dir', str(HARD_DIR),
    '--distilled-dir', str(DISTILLED_DIR),
    '--device', 'cuda',
    '--batch-size', '16',
    '--pair-chunk-size', '2048',
)

## Run the single locked six-model test comparison

In [ ]:
FINAL_RESULTS = RESULT_DIR / 'final_test_results.json'
run_project(
    'src/evaluation/final_comparison.py',
    '--selection', str(SELECTION),
    '--test-data', str(TEST_DATA),
    '--teacher-cache', str(TEACHER_DIR / 'teacher_test_scores.jsonl'),
    '--hard-dir', str(HARD_DIR),
    '--distilled-dir', str(DISTILLED_DIR),
    '--device', 'cuda',
    '--output', str(FINAL_RESULTS),
)
print(FINAL_RESULTS.read_text())

## Benchmark efficiency on this same GPU

BGE candidate precomputation is timed separately. Its online rank-100 measurement uses precomputed normalized candidate embeddings. Qwen must cross-encode all 100 pairs.

In [ ]:
EFFICIENCY_RESULTS = RESULT_DIR / 'efficiency_results.json'
run_project(
    'src/evaluation/benchmark_efficiency.py',
    '--selection', str(SELECTION),
    '--test-data', str(TEST_DATA),
    '--quality-results', str(FINAL_RESULTS),
    '--hard-dir', str(HARD_DIR),
    '--distilled-dir', str(DISTILLED_DIR),
    '--device', 'cuda',
    '--output', str(EFFICIENCY_RESULTS),
)
print(EFFICIENCY_RESULTS.read_text())

## Download the small review records

In [ ]:
review_zip = Path('/content/milestones11_12_results.zip')
with ZipFile(review_zip, 'w', compression=ZIP_DEFLATED) as archive:
    archive.write(SELECTION, 'final_selection.json')
    archive.write(FINAL_RESULTS, 'final_test_results.json')
    archive.write(EFFICIENCY_RESULTS, 'efficiency_results.json')
    archive.writestr('source_bundle_sha256.txt', bundle_hash)
files.download(str(review_zip))